In [1]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments
from tqdm.notebook import tqdm
import pandas as pd
from datasets import Dataset, load_metric, DatasetDict

model = BartForConditionalGeneration.from_pretrained('./fine-tuned-bart-salience')
tokenizer = BartTokenizer.from_pretrained('./fine-tuned-bart-salience')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("device ",device)
model = model.to(device)

2024-08-20 22:02:24.219040: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-20 22:02:24.255299: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-20 22:02:24.999162: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


device  cuda


In [2]:
# Example function to generate a question based on input context, question, label, and salient sentence
def generate_question(context, question, sal):
    # Prepare the input text in the same format as during training
    input_text = f"""Context: {context}

    Question: {question}

    Salient Sentence: {sal}

    Task: Generate a new question based on the context and salient sentence that can be answered with the given context, given the issue."""
    
    # Tokenize the input text
    inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    # Generate the output using the model
    outputs = model.generate(inputs, max_length=128, num_beams=5, early_stopping=True)
    
    # Decode the output text
    generated_question = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return generated_question


In [3]:
mapping = {"N":"negation", "E":"entity swap", "#":"number swap", 
           "A":"to antonym", "I":"no information", 
           "X":"mutual exclusion"}

In [4]:
df_test= pd.read_csv('dev_sal.csv')

def write(res_eval):
    with open('bart_epoch3.txt', 'w') as f:
        for line in res_eval:
            f.write(f"{line}\n")
        f.close()
        
input_seq_eval = []
# labels_seq_eval = []
for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    #label = row['label']
    context = row['context']
    question = row['original_question']
    sal = row['answer_sentence']
    
    input_s = generate_question(context, question, sal)

    input_seq_eval.append(input_s)
    write(input_seq_eval)

  0%|          | 0/5945 [00:00<?, ?it/s]

In [5]:
df_test["BART_ANS"] = input_seq_eval
df_test.to_csv("after_3_epochs.csv")

In [6]:
df_test.to_csv("MRC Eval/after_3_epochs.csv")
!mv bart_epoch3.txt "MRC Eval/"
df_test.to_csv("MRC Eval/SG-Net/after_3_epochs.csv")